This notebook attempts to load existing trained siamese backbones, add a new classification head, then attempt to tune for classification.

In [1]:
# Import helper code
import sys
sys.path.insert(1, '../')
import helpers
import torch

In [35]:
import os
dataset_path = "../split_node21_sets/"
process = "arch_seg" # "crop" "lung_seg" 
train_set = "padchest"
test1_set = "chestxray14"
test2_set = "jsrt"
bsz = 64
resize_dim = 224
# run_index = 0
# best_epoch_to_load = 50
# run_index = 1
# best_epoch_to_load = 46
run_index = 2
best_epoch_to_load = 50
weights = torch.load(f'logs/subsets/{train_set}/rad_unfrz_cosine_{process}_{run_index}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')
# weights = torch.load(f'logs/subsets/chestxray14/rad_unfrz_cosine_{process}_{run_index}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')
# weights = torch.load('logs/subsets/chestxray14/rad_unfrz_cosine_crop_0/checkpoints/best_model.pth',map_location=device)

from torchvision import transforms
import torch
base_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.ToTensor(),
])

augment_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])
# Load with default settings
train_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "train"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=augment_transform,
    cache_in_ram=True

)

Loading dataset: 100%|██████████| 2/2 [00:02<00:00,  1.48s/it]


Caching base images into RAM...


Caching: 100%|██████████| 2348/2348 [00:29<00:00, 78.67it/s] 

Cached 2348 images
Total pairs: 1174
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 219
  normal (idx=0): 955


In [36]:
test_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  2.09it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1008/1008 [00:15<00:00, 66.39it/s]

Cached 1008 images
Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410


In [37]:
test2_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,test1_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  2.70it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1062/1062 [00:15<00:00, 69.48it/s]

Cached 1062 images
Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351


In [38]:
test3_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,test2_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  6.44it/s]


Caching base images into RAM...


Caching: 100%|██████████| 144/144 [00:01<00:00, 72.63it/s]

Cached 144 images
Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28


In [39]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [40]:
model.load_state_dict(weights['model_state_dict'])

<All keys matched successfully>

## Create Classifier Head

In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseClassifier(nn.Module):
    """
    Wraps trained SiameseNetwork with a classification head
    Default freeze the Siamese backbone for fast fine-tuning
    """
    def __init__(self, siamese_model, embedding_dim=128, freeze_siamese=True):
        super(SiameseClassifier, self).__init__()
        
        self.siamese = siamese_model
        
        # Freeze the siamese network if desired
        if freeze_siamese:
            for param in self.siamese.parameters():
                param.requires_grad = False
        
        # Classification head operates on distance/concatenated embeddings
        # Use both distance + embeddings
        
        input_dim = 2 * embedding_dim + 1  # concatenated embeddings + distance
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Binary classification (sigmoid applied later)
        )
        
    def forward(self, x1, x2, return_embeddings=False, distance_metric='cosine'):
        """
        Args:
            x1, x2: Input image pairs
            return_embeddings: If True, also return embeddings for analysis
            distance_metric: 'euclidean' or 'cosine'
        Returns:
            logits: Classification logits (before sigmoid)
            (optional) emb1, emb2, distance
        """
        # Get embeddings from siamese network
        emb1, emb2 = self.siamese(x1, x2)
        
        # Calculate distance
        if distance_metric == 'euclidean':
            distance = F.pairwise_distance(emb1, emb2, p=2)
        elif distance_metric == 'cosine':
            cosine_sim = torch.sum(emb1 * emb2, dim=1)
            distance = 1 - cosine_sim
        else:
            raise ValueError(f"Unknown distance metric: {distance_metric}")
        
        # Concatenate embeddings and distance
        # Shape: (batch, 2*embedding_dim + 1)
        combined = torch.cat([emb1, emb2, distance.unsqueeze(1)], dim=1)
        
        # Get classification logits
        logits = self.classifier(combined).squeeze()
        
        if return_embeddings:
            return logits, emb1, emb2, distance
        return logits

In [42]:
classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

In [43]:
optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

In [44]:
criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [45]:
# import training metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

In [46]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  {test1_set}   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  {test2_set}   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'padchest_heads/rad_{train_set}_cosine_{process}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5372 | Acc: 0.8688 | Prec: 0.6199 | Rec: 0.7671 | F1: 0.6857 | AUC: 0.9158
  Test   Loss: 0.4227 | Acc: 0.8452 | Prec: 0.8636 | Rec: 0.2021 | F1: 0.3276 | AUC: 0.7576
  chestxray14   Loss: 0.5914 | Acc: 0.7081 | Prec: 0.7273 | Rec: 0.2222 | F1: 0.3404 | AUC: 0.7351
  jsrt   Loss: 1.1241 | Acc: 0.3889 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | AUC: 0.5584
  ✓ Saved new best model! (F1: 0.3276)
Epoch 2/20
  Train Loss: 0.1744 | Acc: 0.9523 | Prec: 0.9939 | Rec: 0.7489 | F1: 0.8542 | AUC: 0.9830
  Test   Loss: 0.5354 | Acc: 0.8571 | Prec: 0.8235 | Rec: 0.2979 | F1: 0.4375 | AUC: 0.7468
  chestxray14   Loss: 0.8922 | Acc: 0.7307 | Prec: 0.7176 | Rec: 0.3389 | F1: 0.4604 | AUC: 0.7208
  jsrt   Loss: 2.4507 | Acc: 0.3889 | Prec: 0.5000 | Rec: 0.0455 | F1: 0.0833 | AUC: 0.6112
  ✓ Saved new best model! (F1: 0.4375)
Epoch 3/20
  Train Loss: 0.0965 | Acc: 0.9693 | Prec: 0.9598 | Rec: 0.8721 | F1: 0.9139 | AUC: 0.9874
  Test   Loss: 0.5305 | Acc: 0.8631 | Prec: 0.7907

In [47]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  {test1_set}   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  {test2_set}   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'padchest_heads/rad_{train_set}_cosine_{process}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5459 | Acc: 0.7956 | Prec: 0.4683 | Rec: 0.7078 | F1: 0.5636 | AUC: 0.8512
  Test   Loss: 0.4127 | Acc: 0.8393 | Prec: 0.8421 | Rec: 0.1702 | F1: 0.2832 | AUC: 0.7503
  chestxray14   Loss: 0.6013 | Acc: 0.7043 | Prec: 0.7447 | Rec: 0.1944 | F1: 0.3084 | AUC: 0.7283
  jsrt   Loss: 1.1154 | Acc: 0.3889 | Prec: 0.0000 | Rec: 0.0000 | F1: 0.0000 | AUC: 0.5731
  ✓ Saved new best model! (F1: 0.2832)
Epoch 2/20
  Train Loss: 0.1694 | Acc: 0.9429 | Prec: 0.9750 | Rec: 0.7123 | F1: 0.8232 | AUC: 0.9808
  Test   Loss: 0.5147 | Acc: 0.8651 | Prec: 0.8421 | Rec: 0.3404 | F1: 0.4848 | AUC: 0.7592
  chestxray14   Loss: 0.9083 | Acc: 0.7269 | Prec: 0.7011 | Rec: 0.3389 | F1: 0.4569 | AUC: 0.7149
  jsrt   Loss: 2.3663 | Acc: 0.3750 | Prec: 0.3333 | Rec: 0.0227 | F1: 0.0426 | AUC: 0.6015
  ✓ Saved new best model! (F1: 0.4848)
Epoch 3/20
  Train Loss: 0.0901 | Acc: 0.9744 | Prec: 0.9610 | Rec: 0.8995 | F1: 0.9292 | AUC: 0.9874
  Test   Loss: 0.5322 | Acc: 0.8651 | Prec: 0.8095

In [ ]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)

# contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.load_state_dict(weights['model_state_dict'])

classifier = SiameseClassifier(model, embedding_dim=128, freeze_siamese=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  {test1_set}   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  {test2_set}   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'padchest_heads/rad_{train_set}_cosine_{process}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")